# Анализ email-эксперимента

Я хочу проверить, увеличивают ли рассылки покупки и выручку относительно отсутствия письма. Это учебный анализ открытого эксперимента Kevin Hillstrom, а не мой собственный запуск кампании.

[Источник](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html)

## 1. Загружаю данные

Одна строка соответствует клиенту. Есть три случайно сформированные группы, а результаты измерялись две недели после рассылки.

In [ ]:
from pathlib import Path
import sys
import sqlite3
import pandas as pd
from IPython.display import display, Image

root = Path.cwd()
if not (root / 'data/raw/hillstrom.csv').exists():
    root = root.parent
sys.path.insert(0, str(root))

from src.analysis import load_data, summarize, compare_groups, baseline_balance, diagnostics
from src.plots import create_figures

df = load_data(root / 'data/raw/hillstrom.csv')
display(df.head())
print(df.shape)

## 2. Проверяю качество

Совпадающие строки не удаляю: в наборе нет ID клиента. Проверки типов и согласованности покупок выполняются в `load_data`.

In [ ]:
df.info()
display(df.isna().sum().to_frame('missing'))
print('Совпадающие строки:', df.duplicated().sum())
display(df['segment'].value_counts().to_frame('customers'))

## 3. План сравнения

Основная метрика — конверсия в покупку. Сравниваю каждую рассылку с контролем. Для двух основных сравнений использую Holm. Посещения и выручка — дополнительные метрики. План выбран для учебного ретроспективного анализа; предварительной регистрации до эксперимента не было.

In [ ]:
summary = summarize(df)
display(summary)

## 4. Сверяю Python и SQL

Те же показатели считаю по исходным клиентам в SQLite. Запросы также используют синтаксис, совместимый с PostgreSQL.

In [ ]:
with sqlite3.connect(':memory:') as connection:
    df.to_sql('customers', connection, index=False)
    query = (root / 'sql/01_group_metrics.sql').read_text()
    sql_metrics = pd.read_sql_query(query, connection).set_index('segment')
    display(sql_metrics)
    display(pd.read_sql_query((root / 'sql/02_effects_vs_control.sql').read_text(), connection))

## 5. Проверяю группы до воздействия

Смотрю размеры групп и стандартизованные различия исходных признаков. Небольшие различия не доказывают корректность рандомизации, но помогают заметить явную проблему.

In [ ]:
balance = baseline_balance(df)
checks = diagnostics(df, summary, balance)
print('p-value размеров групп:', checks['group_size_p'])
print('Максимальный абсолютный SMD:', checks['max_abs_smd_checked_baselines'])
display(balance.assign(abs_smd=balance.smd.abs()).sort_values('abs_smd', ascending=False).head(10))

## 6. Оцениваю эффект

Для конверсии и посещений использую двусторонний z-тест. 95% интервалы разницы долей — нормальное приближение без поправки на множественность. Для выручки делаю 30 000 bootstrap-выборок клиентов, включая нулевые покупки, seed 20261111.

В таблице effect для бинарных метрик выражен долей: умножение на 100 даёт процентные пункты. Для spend единица — доллар на клиента. p-value выручки рассчитан отдельным асимптотическим тестом разницы средних. Поправка Holm применяется внутри каждой метрики; основной подтверждающий вывод относится к конверсии.

In [ ]:
effects = compare_groups(df)
display(effects)
conversion = effects[effects.metric == 'conversion'].copy()
conversion['effect_pp'] = conversion.effect * 100
conversion['ci_low_pp'] = conversion.ci95_low * 100
conversion['ci_high_pp'] = conversion.ci95_high * 100
display(conversion[['treatment', 'effect_pp', 'relative_lift', 'ci_low_pp', 'ci_high_pp', 'p_holm_within_metric']])

## 7. Показываю результат

Интервалы помогают увидеть точность оценки, а не только направление разницы.

In [ ]:
create_figures(summary, effects, root / 'reports/figures')
display(Image(filename=str(root / 'reports/figures/overview.png')))

## 8. Перевожу эффект в понятный масштаб

Это ожидаемый дополнительный результат на 10 000 сопоставимых клиентов за две недели. Он не является новым фактически проведённым экспериментом.

In [ ]:
audience = 10000
projected = effects[effects.metric.isin(['conversion', 'spend'])].copy()
projected['increment_per_10000'] = projected.effect * audience
projected['ci95_low_per_10000'] = projected.ci95_low * audience
projected['ci95_high_per_10000'] = projected.ci95_high * audience
display(projected[['metric', 'treatment', 'increment_per_10000', 'ci95_low_per_10000', 'ci95_high_per_10000']])

## Вывод

Обе рассылки увеличили конверсию относительно контроля. Для Mens оценка прироста составляет около 0,681 п. п., для Womens — 0,311 п. п. Mens выглядит более перспективным общим вариантом.

Выручка на клиента тоже выросла, но прибыльность я не оцениваю: нет маржи и стоимости кампаний. Исторические данные не гарантируют такой же результат на современной аудитории. Для новой кампании нужен новый эксперимент.

Сегментные разрезы доступны в Superset и CSV. Я использую их для поиска будущих гипотез, а не для обещания точного эффекта персонального таргетинга.